[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SEU_USUARIO/SEU_REPOSITORIO/blob/main/src/JonattasFelipe_rm572692_pbl_fase6.ipynb)

# FarmTech Solutions – Fase 6: Visão Computacional com YOLO

**Grupo:**
- Jonattas Felipe – RM572692
- Natanael Filho – RM572474
- Pedro Henrique – RM571394
- Andrews Oliveira – RM572311

Notebook organizado em:
1. Preparação do dataset (FiftyOne / Open Images, ou dataset pronto do GitHub)
2. Rotulação no Make Sense
3. Entrega 1: YOLOv5 customizado (30 e 60 épocas)
4. Entrega 2: YOLO padrão e CNN do zero, com comparação final
5. Conclusões

**Objetos escolhidos:** Trator (A) e Vaca/gado (B), alinhados ao contexto agro da FarmTech.

### Modelos de linguagem (IA generativa) utilizados no projeto

| Modelo | Versão | Desenvolvedor | Uso no projeto |
|---|---|---|---|
| Claude Sonnet | 5.5 | Anthropic | Revisão do notebook e do README, sugestões de correção de código e apoio na redação das conclusões automáticas |

Todo o código foi executado e validado pelo grupo no Google Colab. As versões das bibliotecas efetivamente usadas na execução são impressas na célula "Versões do ambiente" (seção 3).


## 1. Preparação do dataset

Para montar o banco de imagens usei a biblioteca **FiftyOne**, que baixa imagens do dataset público **Open Images V7**. Baixo apenas as **imagens**: a rotulação das caixas será feita por mim no Make Sense (etapa 2).

Passos: baixar candidatas de cada classe, manter só imagens com **um único objeto da classe** (facilita a rotulação e evita ambiguidade), sortear 40 por classe e dividir em **32 treino / 4 validação / 4 teste**.

In [ ]:
# Instala a FiftyOne (pode levar 1-2 minutos)
!pip install -q fiftyone

In [ ]:
# Conecta ao Google Drive para guardar o dataset
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/FarmTech_Fase6'   # pasta do projeto no Drive
!mkdir -p {BASE}

In [ ]:
import random, shutil
from pathlib import Path
from PIL import Image
import fiftyone.zoo as foz

random.seed(42)  # reprodutibilidade do sorteio

# Classe do Open Images -> nome usado no projeto
CLASSES = {'Tractor': 'trator', 'Cattle': 'vaca'}
N_POR_CLASSE = 40
SPLITS = {'train': 32, 'val': 4, 'test': 4}

# Estrutura de pastas no formato esperado pelo YOLOv5
DATASET = Path(BASE) / 'dataset'
for s in SPLITS:
    (DATASET / 'images' / s).mkdir(parents=True, exist_ok=True)
    (DATASET / 'labels' / s).mkdir(parents=True, exist_ok=True)

In [ ]:
# Reprodutibilidade: se o repositório do GitHub já traz o dataset rotulado (images/ + labels/),
# usamos ele e PULAMOS o download aleatório do Open Images (evita rótulos desalinhados com as imagens).
REPO_URL = 'https://github.com/SEU_USUARIO/SEU_REPOSITORIO.git'   # <- troque pelo link do repositório do grupo
!rm -rf /content/repo && git clone -q {REPO_URL} /content/repo

lab_repo = Path('/content/repo/dataset/labels/train')
DATASET_PRONTO = lab_repo.exists() and any(lab_repo.glob('*.txt'))
if DATASET_PRONTO:
    shutil.copytree('/content/repo/dataset', DATASET, dirs_exist_ok=True)
    print('Dataset rotulado carregado do GitHub. Seções de download e de rotulação serão puladas.')
else:
    print('Dataset não encontrado no GitHub: seguindo o fluxo completo (FiftyOne + Make Sense).')


In [ ]:
candidatas = {} if DATASET_PRONTO else {nome: baixar_e_filtrar(c, nome) for c, nome in CLASSES.items()}


In [ ]:
def salvar(caminho_origem, destino, lado_max=640):
    """Redimensiona (lado maior = 640 px) e salva em JPG."""
    img = Image.open(caminho_origem).convert('RGB')
    img.thumbnail((lado_max, lado_max))
    img.save(destino, 'JPEG', quality=92)

contagem = {}
if not DATASET_PRONTO:
    for nome, caminhos in candidatas.items():
        assert len(caminhos) >= N_POR_CLASSE, f'Poucas imagens de {nome}; aumente max_samples'
        escolhidas = random.sample(caminhos, N_POR_CLASSE)
        i = 0
        for split, qtd in SPLITS.items():
            for caminho in escolhidas[i:i + qtd]:
                i += 1
                salvar(caminho, DATASET / 'images' / split / f'{nome}_{i:02d}.jpg')
            contagem[(nome, split)] = qtd

for s in SPLITS:
    print(s, len(list((DATASET / 'images' / s).glob('*.jpg'))), 'imagens')

In [ ]:
# Confere visualmente uma amostra de cada classe
import matplotlib.pyplot as plt
fig, eixos = plt.subplots(2, 4, figsize=(14, 7))
for linha, nome in enumerate(CLASSES.values()):
    fotos = sorted((DATASET / 'images' / 'train').glob(f'{nome}_*.jpg'))[:4]
    for ax, f in zip(eixos[linha], fotos):
        ax.imshow(Image.open(f)); ax.set_title(f.name); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
if not DATASET_PRONTO:
    shutil.make_archive(f'{BASE}/imagens_para_rotular', 'zip', DATASET / 'images')
    print('ZIP salvo em:', f'{BASE}/imagens_para_rotular.zip')


**Fonte das imagens:** Open Images V7 (Google), acessado via FiftyOne. Imagens usadas apenas para fins acadêmicos.

## 2. Rotulação no Make Sense

1. Acessar https://www.makesense.ai, clicar em *Get Started* e carregar as imagens de `images/train` (depois repetir o processo para `val` e `test`).
2. Escolher *Object Detection* e criar as labels **nesta ordem**: `trator` (id 0) e `vaca` (id 1).
3. Desenhar uma caixa ao redor do objeto em cada imagem e associar a label.
4. Em *Actions > Export Annotations*, escolher **A .zip package containing files in YOLO format**.
5. Salvar os três ZIPs no Drive com estes nomes: `labels_zips/train.zip`, `labels_zips/val.zip` e `labels_zips/test.zip`.

A célula abaixo extrai os `.txt` para a pasta `labels/` correta e confere se todas as imagens têm rótulo.

> Se o repositório do GitHub já contém `dataset/labels/`, a célula 1 detecta isso e **pula** a rotulação e a extração abaixo. Os ZIPs só são necessários quando o dataset é refeito do zero.


In [ ]:
# Extrai os rótulos exportados do Make Sense para labels/train, val e test
for s in ([] if DATASET_PRONTO else SPLITS):
    zip_path = f'{BASE}/labels_zips/{s}.zip'
    destino = DATASET / 'labels' / s
    !unzip -o -j -q "{zip_path}" -d "{destino}"

# Confere: toda imagem precisa ter um .txt de mesmo nome, com classe 0 ou 1
for s in SPLITS:
    imgs = sorted((DATASET / 'images' / s).glob('*.jpg'))
    sem_label = [i.name for i in imgs if not (DATASET / 'labels' / s / (i.stem + '.txt')).exists()]
    classes_usadas = set()
    for t in (DATASET / 'labels' / s).glob('*.txt'):
        for linha in t.read_text().splitlines():
            if linha.strip():
                classes_usadas.add(int(linha.split()[0]))
    print(f'{s}: {len(imgs)} imagens | sem rótulo: {sem_label} | classes: {classes_usadas}')

## 3. Entrega 1 – YOLOv5 customizado

**Passo a passo:**
1. **Preparo:** copiar o dataset para o disco local do Colab (o treino fica bem mais rápido que lendo do Drive) e criar o `data.yaml`.
2. **Treino:** *fine-tuning* do `yolov5s.pt` (pré-treinado no COCO) com nossas duas classes, em duas simulações: **30 e 60 épocas**.
3. **Validação:** curvas de erro (loss), precisão, recall e mAP calculadas pelo YOLO a cada época sobre a pasta `val`.
4. **Teste:** avaliação do `best.pt` nas imagens da pasta `test`, nunca vistas no treino, e visualização das detecções.

In [ ]:
import time, shutil, pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Copia o dataset para o disco local do Colab (treino mais rápido)
LOCAL = Path('/content/dataset_local')
shutil.copytree(DATASET, LOCAL, dirs_exist_ok=True)

# Arquivo de configuração do dataset para o YOLOv5
yaml_texto = '''path: /content/dataset_local
train: images/train
val: images/val
test: images/test
nc: 2
names: ['trator', 'vaca']
'''
Path('/content/data.yaml').write_text(yaml_texto)
shutil.copy('/content/data.yaml', f'{BASE}/data.yaml')

# Clona o YOLOv5 e instala dependências
!git clone -q https://github.com/ultralytics/yolov5 /content/yolov5
%cd /content/yolov5
!pip install -q -r requirements.txt

In [ ]:
# Versões do ambiente (registradas automaticamente para reprodutibilidade)
import sys, subprocess
from importlib.metadata import version, PackageNotFoundError

print('Python:', sys.version.split()[0])
for pacote in ['torch', 'tensorflow', 'fiftyone', 'pandas', 'numpy', 'matplotlib', 'scikit-learn', 'pillow']:
    try:
        print(f'{pacote}: {version(pacote)}')
    except PackageNotFoundError:
        print(f'{pacote}: não instalado')
commit = subprocess.run(['git', '-C', '/content/yolov5', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('YOLOv5 (ultralytics/yolov5) commit:', commit, '| pesos iniciais: yolov5s.pt (COCO)')


In [ ]:
# Simulação 1: 30 épocas
t0 = time.time()
!python train.py --img 640 --batch 8 --epochs 30 --data /content/data.yaml --weights yolov5s.pt --name exp_30ep --exist-ok
tempo_treino_30 = time.time() - t0
print(f'Tempo de treino (30 épocas): {tempo_treino_30/60:.1f} min')

In [ ]:
# Simulação 2: 60 épocas
t0 = time.time()
!python train.py --img 640 --batch 8 --epochs 60 --data /content/data.yaml --weights yolov5s.pt --name exp_60ep --exist-ok
tempo_treino_60 = time.time() - t0
print(f'Tempo de treino (60 épocas): {tempo_treino_60/60:.1f} min')

### 3.1 Validação: curvas de treino

O YOLO registra em `results.csv` as métricas de cada época. Abaixo comparo as duas simulações.

In [ ]:
def ler_resultados(nome):
    df = pd.read_csv(f'/content/yolov5/runs/train/{nome}/results.csv')
    df.columns = df.columns.str.strip()
    return df

res = {'30 épocas': ler_resultados('exp_30ep'), '60 épocas': ler_resultados('exp_60ep')}

# Métricas da última época de cada simulação (validação)
linhas = []
for nome, df in res.items():
    u = df.iloc[-1]
    linhas.append({'Simulação': nome,
                   'Precisão': u['metrics/precision'], 'Recall': u['metrics/recall'],
                   'mAP@0.5': u['metrics/mAP_0.5'], 'mAP@0.5:0.95': u['metrics/mAP_0.5:0.95'],
                   'Loss val (box)': u['val/box_loss']})
tabela_val = pd.DataFrame(linhas).round(3)
display(tabela_val)

# Gráficos comparativos
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for nome, df in res.items():
    ax[0].plot(df['train/box_loss'], label=nome)
    ax[1].plot(df['val/box_loss'], label=nome)
    ax[2].plot(df['metrics/mAP_0.5'], label=nome)
for a, t in zip(ax, ['Erro de treino (box loss)', 'Erro de validação (box loss)', 'mAP@0.5 na validação']):
    a.set_title(t); a.set_xlabel('época'); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Escolha do melhor modelo pela VALIDAÇÃO (nunca pelo teste): maior mAP@0.5, desempate por mAP@0.5:0.95
ordem = tabela_val.sort_values(['mAP@0.5', 'mAP@0.5:0.95'], ascending=False, kind='stable')
melhor_nome = ordem.iloc[0]['Simulação']
EXP = {'30 épocas': ('exp_30ep', tempo_treino_30), '60 épocas': ('exp_60ep', tempo_treino_60)}
melhor_exp, melhor_tempo = EXP[melhor_nome]
print(f'Melhor modelo na validação: {melhor_nome} ({melhor_exp})')


### 3.2 Teste: métricas e imagens processadas

Avaliação do `best.pt` de cada simulação nas 8 imagens da pasta `test`.

In [ ]:
resultados_teste = {}
for nome, exp in [('30 épocas', 'exp_30ep'), ('60 épocas', 'exp_60ep')]:
    saida = !python val.py --weights runs/train/{exp}/weights/best.pt --data /content/data.yaml --task test --img 640 --name test_{exp} --exist-ok 2>&1
    linha = [l for l in saida if l.strip().startswith('all')][0].split()
    P, R, m50, m95 = map(float, linha[3:7])
    resultados_teste[nome] = {'Precisão': P, 'Recall': R, 'mAP@0.5': m50, 'mAP@0.5:0.95': m95}
    print(nome, resultados_teste[nome])

display(pd.DataFrame(resultados_teste).T.round(3))

In [ ]:
# Detecção nas imagens de teste (resultados em runs/detect/detect_XXep)
for exp in ['exp_30ep', 'exp_60ep']:
    !python detect.py --weights runs/train/{exp}/weights/best.pt --source /content/dataset_local/images/test --img 640 --conf-thres 0.25 --name detect_{exp} --exist-ok

# Mostra as imagens processadas pelo modelo de 60 épocas
fotos = sorted(Path('/content/yolov5/runs/detect/detect_exp_60ep').glob('*.jpg'))
fig, eixos = plt.subplots(2, 4, figsize=(18, 9))
for ax, f in zip(eixos.ravel(), fotos):
    ax.imshow(Image.open(f)); ax.set_title(f.name); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Mostra também o modelo de 30 épocas, para comparar visualmente
fotos = sorted(Path('/content/yolov5/runs/detect/detect_exp_30ep').glob('*.jpg'))
fig, eixos = plt.subplots(2, 4, figsize=(18, 9))
for ax, f in zip(eixos.ravel(), fotos):
    ax.imshow(Image.open(f)); ax.set_title(f.name); ax.axis('off')
plt.tight_layout(); plt.show()

# Salva todos os resultados no Drive
shutil.copytree('/content/yolov5/runs', f'{BASE}/runs', dirs_exist_ok=True)
print('Resultados salvos em', f'{BASE}/runs')

## 4. Entrega 2 – Comparação de abordagens

Comparo três abordagens sobre a **mesma base de teste**:

| Abordagem | Descrição |
|---|---|
| **A. YOLO customizado** | YOLOv5s com fine-tuning nas nossas classes (Entrega 1) |
| **B. YOLO padrão** | YOLOv5s com pesos originais do COCO, **sem treinar**. O COCO tem a classe `cow`, mas **não tem `trator`**; usei `truck` como equivalente aproximado |
| **C. CNN do zero** | Rede convolucional pequena treinada só com nossas 64 imagens, classificando a imagem inteira |

**Métrica comum:** acurácia por imagem. Nos YOLOs, a classe da detecção com maior confiança define a classe da imagem; sem detecção conta como erro. Também meço o **tempo médio de inferência** por imagem.

In [ ]:
import torch
DISPOSITIVO = 'GPU ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('Dispositivo de inferência:', DISPOSITIVO)

pasta_teste = LOCAL / 'images' / 'test'
arquivos_teste = sorted(pasta_teste.glob('*.jpg'))

def avaliar_yolo(modelo, mapa=None):
    modelo.conf = 0.25
    modelo(str(arquivos_teste[0]))                       # aquecimento (não entra na medida)
    acertos, tempos, previsoes = 0, [], []
    for f in arquivos_teste:
        verdade = f.name.split('_')[0]
        t0 = time.perf_counter()
        r = modelo(str(f), size=640)
        tempos.append(time.perf_counter() - t0)
        df = r.pandas().xyxy[0]
        if mapa is not None:                             # traduz classes do COCO para as nossas
            df = df[df['name'].isin(mapa)].copy()
            df['name'] = df['name'].map(mapa)
        pred = df.sort_values('confidence', ascending=False)['name'].iloc[0] if len(df) else 'nenhum'
        previsoes.append((f.name, verdade, pred))
        acertos += (pred == verdade)
    return acertos / len(arquivos_teste), np.mean(tempos) * 1000, previsoes

# A. YOLO customizado: modelo escolhido pela validação (célula 3.1)
yolo_custom = torch.hub.load('/content/yolov5', 'custom', path=f'/content/yolov5/runs/train/{melhor_exp}/weights/best.pt', source='local')
acc_A, ms_A, prev_A = avaliar_yolo(yolo_custom)

# B. YOLO padrão (COCO): cow -> vaca, truck -> trator (aproximação)
yolo_padrao = torch.hub.load('/content/yolov5', 'yolov5s', source='local')
acc_B, ms_B, prev_B = avaliar_yolo(yolo_padrao, mapa={'cow': 'vaca', 'truck': 'trator'})

print(f'A) YOLO customizado: acurácia {acc_A:.0%} | {ms_A:.1f} ms/imagem')
print(f'B) YOLO padrão:      acurácia {acc_B:.0%} | {ms_B:.1f} ms/imagem')
print('\nPrevisões do YOLO padrão (arquivo, verdade, previsto):')
for p in prev_B: print(p)

### 4.1 CNN treinada do zero

Rede pequena (3 blocos convolucionais) com *data augmentation* (espelhamento, rotação e zoom), porque 64 imagens de treino é muito pouco. Ela só diz **qual classe** a imagem tem; não localiza o objeto.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

LADO = 128

def carregar(split):
    X, y = [], []
    for f in sorted((LOCAL / 'images' / split).glob('*.jpg')):
        X.append(np.array(Image.open(f).convert('RGB').resize((LADO, LADO))))
        y.append(1 if f.name.startswith('vaca') else 0)     # 0 = trator, 1 = vaca
    return np.array(X, dtype='float32'), np.array(y)

Xtr, ytr = carregar('train'); Xva, yva = carregar('val'); Xte, yte = carregar('test')
print(Xtr.shape, Xva.shape, Xte.shape)

cnn = models.Sequential([
    layers.Input((LADO, LADO, 3)),
    layers.Rescaling(1/255),                                 # normaliza pixels para 0-1
    layers.RandomFlip('horizontal'), layers.RandomRotation(0.1), layers.RandomZoom(0.15),
    layers.Conv2D(16, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid'),                   # probabilidade de ser vaca
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.summary()

In [ ]:
parar = callbacks.EarlyStopping(patience=12, restore_best_weights=True, monitor='val_loss')

t0 = time.time()
hist = cnn.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=60, batch_size=16, callbacks=[parar], verbose=2)
tempo_treino_cnn = time.time() - t0
print(f'Tempo de treino da CNN: {tempo_treino_cnn:.1f} s')

# Curvas de aprendizado
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist.history['loss'], label='treino'); ax[0].plot(hist.history['val_loss'], label='validação'); ax[0].set_title('Erro (loss)')
ax[1].plot(hist.history['accuracy'], label='treino'); ax[1].plot(hist.history['val_accuracy'], label='validação'); ax[1].set_title('Acurácia')
for a in ax: a.legend(); a.grid(alpha=.3); a.set_xlabel('época')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pred_cnn = (cnn.predict(Xte, verbose=0).ravel() > 0.5).astype(int)
acc_C = (pred_cnn == yte).mean()

# Tempo de inferência por imagem (uma a uma, igual aos YOLOs)
cnn.predict(Xte[:1], verbose=0)                              # aquecimento
tempos = []
for i in range(len(Xte)):
    t0 = time.perf_counter(); cnn.predict(Xte[i:i+1], verbose=0); tempos.append(time.perf_counter() - t0)
ms_C = np.mean(tempos) * 1000

print(f'C) CNN do zero: acurácia {acc_C:.0%} | {ms_C:.1f} ms/imagem')
ConfusionMatrixDisplay(confusion_matrix(yte, pred_cnn), display_labels=['trator', 'vaca']).plot()
plt.title('CNN do zero – matriz de confusão (teste)'); plt.show()

### 4.2 Comparação final

In [ ]:
comparacao = pd.DataFrame({
    'Abordagem': [f'A. YOLO customizado ({melhor_nome})', 'B. YOLO padrão (COCO)', 'C. CNN do zero'],
    'Acurácia no teste': [acc_A, acc_B, acc_C],
    'Tempo de treino (min)': [melhor_tempo / 60, 0.0, tempo_treino_cnn / 60],
    'Inferência (ms/imagem)': [ms_A, ms_B, ms_C],
    'Localiza o objeto?': ['Sim', 'Sim', 'Não'],
}).round(3)
display(comparacao)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
nomes = ['A. YOLO\ncustom', 'B. YOLO\npadrão', 'C. CNN\ndo zero']
ax[0].bar(nomes, comparacao['Acurácia no teste']); ax[0].set_title('Acurácia no teste'); ax[0].set_ylim(0, 1.05)
ax[1].bar(nomes, comparacao['Tempo de treino (min)']); ax[1].set_title('Tempo de treino (min)')
ax[2].bar(nomes, comparacao['Inferência (ms/imagem)']); ax[2].set_title('Inferência (ms/imagem)')
plt.tight_layout(); plt.show()

In [ ]:
# Comparação crítica gerada com os números desta execução
from IPython.display import Markdown, display

tA = resultados_teste[melhor_nome]
display(Markdown(f"""
### 4.3 Comparação crítica (números desta execução; inferência em **{DISPOSITIVO}**)

| Critério | A. YOLO customizado ({melhor_nome}) | B. YOLO padrão (COCO) | C. CNN do zero |
|---|---|---|---|
| **Facilidade de uso/integração** | Média: exige rotular caixas, treinar e gerenciar o repositório YOLOv5, mas entrega um `best.pt` pronto para webcam/ESP32-CAM. | Alta: zero treino, poucas linhas de código. | Média: código curto em Keras, mas é preciso projetar a arquitetura e ajustar hiperparâmetros. |
| **Precisão** | Acurácia por imagem **{acc_A:.0%}**; mAP@0.5 no teste **{tA['mAP@0.5']:.3f}** | Acurácia por imagem **{acc_B:.0%}** (`trator` não existe no COCO; `truck` é aproximação) | Acurácia por imagem **{acc_C:.0%}** (só classifica, não localiza) |
| **Tempo de treino** | {melhor_tempo/60:.1f} min | Nenhum (pesos prontos) | {tempo_treino_cnn:.0f} s |
| **Tempo de inferência** | {ms_A:.1f} ms/imagem | {ms_B:.1f} ms/imagem | {ms_C:.1f} ms/imagem |
"""))

ranking = sorted([('A. YOLO customizado', acc_A), ('B. YOLO padrão', acc_B), ('C. CNN do zero', acc_C)], key=lambda x: -x[1])
rapido = min([('A', ms_A), ('B', ms_B), ('C', ms_C)], key=lambda x: x[1])[0]
display(Markdown(f"""
**Leitura dos resultados**
- Maior acurácia por imagem no teste: **{ranking[0][0]}** ({ranking[0][1]:.0%}); menor: **{ranking[-1][0]}** ({ranking[-1][1]:.0%}).
- Menor tempo de inferência: abordagem **{rapido}**. As medidas não são perfeitamente equivalentes: no YOLO o tempo inclui pré e pós-processamento, e na CNN o `predict` do Keras tem overhead próprio por chamada.
- O YOLO padrão não conhece `trator`: o desempenho nessa classe depende de o modelo confundir o trator com `truck`, o que mostra por que o *fine-tuning* é necessário para objetos fora do COCO.
- A CNN só diz **qual** classe a imagem tem; o YOLO também diz **onde** o objeto está e lida com cenas de vários objetos.
- Com apenas 8 imagens de teste, cada erro pesa 12,5 pontos percentuais: os números são indicativos, não conclusivos.
"""))


## 5. Conclusões

In [ ]:
# Conclusões sobre 30 x 60 épocas, geradas a partir de results.csv e das métricas de teste
m30, m60 = res['30 épocas'], res['60 épocas']
v30, v60 = tabela_val.set_index('Simulação').loc['30 épocas'], tabela_val.set_index('Simulação').loc['60 épocas']
t30, t60 = resultados_teste['30 épocas'], resultados_teste['60 épocas']

def sinal_overfit(df):
    """Compara o menor erro de validação com o erro final: alta relevante indica overfitting."""
    menor, final = df['val/box_loss'].min(), df['val/box_loss'].iloc[-1]
    ep = int(df['val/box_loss'].idxmin()) + 1
    return (final > menor * 1.15), ep, menor, final

o30, o60 = sinal_overfit(m30), sinal_overfit(m60)
ganho_val = v60['mAP@0.5'] - v30['mAP@0.5']
ganho_teste = t60['mAP@0.5'] - t30['mAP@0.5']

display(Markdown(f"""
### Efeito do número de épocas (30 × 60)
- **Validação:** mAP@0.5 foi de **{v30['mAP@0.5']:.3f}** (30 ép.) para **{v60['mAP@0.5']:.3f}** (60 ép.), variação de **{ganho_val:+.3f}**; precisão {v30['Precisão']:.3f} → {v60['Precisão']:.3f} e recall {v30['Recall']:.3f} → {v60['Recall']:.3f}.
- **Teste:** mAP@0.5 de **{t30['mAP@0.5']:.3f}** → **{t60['mAP@0.5']:.3f}** ({ganho_teste:+.3f}).
- **Overfitting:** com 30 épocas, o menor erro de validação ocorreu na época {o30[1]} ({o30[2]:.4f}) e terminou em {o30[3]:.4f} — {'há sinal de overfitting' if o30[0] else 'sem sinal relevante de overfitting'}. Com 60 épocas, o menor foi na época {o60[1]} ({o60[2]:.4f}) e terminou em {o60[3]:.4f} — {'há sinal de overfitting' if o60[0] else 'sem sinal relevante de overfitting'}.
- **Custo:** {tempo_treino_30/60:.1f} min contra {tempo_treino_60/60:.1f} min ({tempo_treino_60/tempo_treino_30:.1f}x mais tempo). {'O ganho de mAP justifica o custo extra.' if ganho_val > 0.02 else 'O ganho de mAP é pequeno diante do tempo extra, então o custo adicional compensa pouco.'}
- **Modelo escolhido pela validação:** {melhor_nome}.
"""))


### Resultados no teste
As imagens processadas na seção 3.2 mostram, para cada foto de teste, a caixa predita e a confiança do modelo. Para o cliente da FarmTech, o ponto principal é que o modelo localiza e nomeia `trator` e `vaca` mesmo com apenas 64 imagens de treino, graças ao *fine-tuning* do YOLOv5s pré-treinado. Os casos de caixa mal posicionada, confiança baixa ou objeto não detectado devem ser observados diretamente nessas imagens e citados no vídeo.

### Pontos fortes
- Detecção e localização com poucas imagens graças ao *fine-tuning*.
- Modelo leve (`yolov5s`), adequado para execução em tempo real.
- `best.pt` reutilizável em webcam ou ESP32-CAM (projeto "Ir Além").
- Comparação justa entre as três abordagens sobre a mesma base de teste, com seleção do modelo pela validação.

### Limitações
- Dataset pequeno (80 imagens); validação e teste com 8 imagens cada, o que torna as métricas instáveis.
- A base foi filtrada para um objeto por foto, então o modelo não foi testado em cenas com vários objetos.
- `truck` como substituto de `trator` no YOLO padrão é uma aproximação.
- A acurácia por imagem usada na comparação não equivale ao mAP; ela só mede a classe da detecção mais confiante.
- Tempos de inferência medidos em ambientes e com sobrecargas diferentes (YOLO via `torch.hub` e Keras `predict`).

### Próximos passos
Mais imagens e cenas com múltiplos objetos, *transfer learning* com MobileNet/VGG, e teste do `best.pt` em tempo real com webcam ou ESP32-CAM.